# The views are fused into one scene

Sections 3 and 5.3 (Fig. 3a, Fig. 6a, Table 3): the geometry probe appends learnable tokens to the **frozen** model and decodes
from their states the 3D point of every visual token, in the first image's frame. *Single-view* points are those of the
first image; *fused* points are those of the other images, which can only be placed by relating the views. The probe is
trained the same way on Qwen3.5 (Base) and on the backbone of S-Mind after SFT.

**Needs:** the geometry supervision (`scripts/data/geometry.sh`: `data/smind/probe`, which needs ScanNet and the VGGT points of MindCube); one
GPU. Each probe trains probe tokens on the frozen model and then fits fresh readouts on their states, about three hours
per model on one H100 (four for 9B).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"   # or "9b"
GPU = "0"

In [ ]:
nb.need("data/smind/probe/maps_scannet_dev.parquet")
MODELS = {"Base": nb.base_model(SIZE), "S-Mind (SFT)": nb.checkpoint(SIZE, "sft")}
for name, model in MODELS.items():
    out = nb.RUNS / f"{SIZE}-probe/{name.split()[0].lower()}"
    nb.run(["bash", "analysis/diagnosis/geometry_probe.sh"], out / "run.log", done=out / "geometry_probe.json",
           gpu=GPU, env=dict(MODEL=str(model), OUT=str(out), GPU=GPU))

In [ ]:
METRICS = {"single-view points": ("point_far_v1", "all"), "fused points": ("point_far_vk", "all"),
           "  2 images": ("point_far_vk", "spar2"), "  3 images": ("point_far_vk", "spar3"),
           "  8 images": ("point_far_vk", "spatialladder8"), "cross-view bearing": ("cross_bearing", "all"),
           "camera facing": ("facing", "all")}
PAPER = {"Base": [60.62, 11.23, 19.21, 10.26, 4.88, 34.41, 57.59],
         "S-Mind (SFT)": [86.32, 65.11, 75.15, 64.55, 50.52, 85.35, 93.55]}
import pandas as pd
cols = {}
for name in MODELS:
    r = nb.read_json(nb.RUNS / f"{SIZE}-probe/{name.split()[0].lower()}/geometry_probe.json")["scannet"]["fresh_true"]["dev"]
    cols[name] = {m: 100 * r[sub][key]["acc"] for m, (key, sub) in METRICS.items()}
    if SIZE == "4b":
        cols[f"paper: {name}"] = dict(zip(METRICS, PAPER[name]))
df = pd.DataFrame(cols)
display(df)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
labels = list(METRICS)
for i, name in enumerate(MODELS):
    ax.barh([j + 0.38 * i for j in range(len(labels))], df[name], height=0.36, label=name)
ax.set_yticks([j + 0.19 for j in range(len(labels))], labels)
ax.invert_yaxis(); ax.set_xlabel("% of ScanNet dev points / pairs / cameras"); ax.legend(frameon=False)
plt.tight_layout()

The frozen Qwen3.5 recovers the first image's own points but few of the other images', which need the views related;
S-Mind's backbone recovers most of both, and keeps its lead as images are added.